In [64]:
import sys

import numpy as np
from transforms3d.axangles import axangle2mat, mat2axangle
import torch
import plotly.graph_objects as go
import trimesh

from mano_pybullet.hand_model import HandModel20

In [65]:
sys.path.append("..")

from utils.grasp_utils import get_handmodel
from model.hand_opt import AdamGraspTransfer

In [66]:
def mat2rvec(mat):
    """Convert rotation matrix to rotation vector."""
    axis, angle = mat2axangle(mat, unit_thresh=1e-05)
    return axis * angle

In [67]:
# NOTE: Set the mano hand models dir here. When using with a script, load this directory from a some config file

%env MANO_MODELS_DIR=/home/ninad/Projects/MANO/MANO_Hand_Model/mano_v1_2/models

env: MANO_MODELS_DIR=/home/ninad/Projects/MANO/MANO_Hand_Model/mano_v1_2/models


## Set Sample Data

In [68]:
# Load data for the 00100 frame
# fname = "sample_hamer_output.npz"

# frame_id = "000172"
frame_id = "000247"

fname = f"{frame_id}.npz"


data = np.load(f"../data/{fname}", allow_pickle=True)

In [69]:
for k in data.keys():
  print(k)

pred_cam
pred_mano_params
pred_cam_t
focal_length
pred_keypoints_3d
pred_vertices
pred_keypoints_2d
opt_translation
bboxes
right
target_transfer_pose


In [70]:
data['right']

array([0, 1])

## Set Left/Right

In [71]:
use_left_hand = True
print("Use left hand? -->", use_left_hand)

rl_index = data['right']
left_idxs = np.arange(rl_index.shape[0])[rl_index==0]
right_idxs = np.arange(rl_index.shape[0])[rl_index==1]

print(left_idxs)
print(right_idxs)

idx_to_use = left_idxs if use_left_hand else right_idxs
print(use_left_hand, idx_to_use)

Use left hand? --> True
[0]
[1]
True [0]


In [72]:
print(left_idxs.size, right_idxs.size)

1 1


In [73]:
mano_params = data['pred_mano_params'].item()
print(type(mano_params))
print(mano_params.keys())
print(mano_params['hand_pose'].shape) # for 2 hands

<class 'dict'>
dict_keys(['global_orient', 'hand_pose', 'betas'])
(2, 15, 3, 3)


In [74]:
hand_rotn_mat = mano_params['global_orient'][idx_to_use][0][0]
hand_theta_mat = mano_params['hand_pose'][idx_to_use][0]
mano_trans = data['opt_translation'][idx_to_use][0]
print(hand_rotn_mat.shape)
print(hand_theta_mat.shape)
print(mano_trans.shape)

(3, 3)
(15, 3, 3)
(3,)


In [75]:
hand_theta_full = np.array([mat2rvec(hand_rotn_mat)] + [mat2rvec(hand_theta_mat[i]) for i in range(hand_theta_mat.shape[0])])
print(hand_theta_full.shape)

(16, 3)


## Init Gripper Models

In [91]:
# source_gripper = "mano_left" if use_left_hand else "mano_right"
source_gripper = "mano_right"
target_gripper = "fetch_gripper"
device = "cpu"

In [92]:
source_model = get_handmodel(
  source_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

In [93]:
target_model = get_handmodel(
  target_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

## Grasp Pose

In [94]:
# Mano Pybullet Model
# hand_model = HandModel20(left_hand=use_left_hand)
hand_model = HandModel20(left_hand=False)
# hm_left = HandModel20(left_hand=True)

angles, palm_basis = hand_model.mano_to_angles(hand_theta_full)
print(len(angles))
print(palm_basis)

# Reference: https://github.com/kninad/mano_pybullet/blob/960c257cf465f8966e770562b66150beaa359230/mano_pybullet/hand_body.py#L155
origin = hand_model.origins()[0]
# origin = hm_left.origins()[0]

palm_trans = mano_trans + origin - palm_basis @ origin

print(palm_trans.shape, palm_trans)
print(mano_trans)

# palm_trans = np.array([palm_trans[0], mano_trans[1], mano_trans[2]])

20
[[-0.1285984  -0.40649506  0.90455747]
 [-0.91437279 -0.30450351 -0.26683329]
 [ 0.38390734 -0.86141708 -0.33252936]]
(3,) [0.94793033 0.46640419 1.31312467]
[0.84295843 0.36894829 1.33611083]


In [105]:
# SOURCE GRIPPER (MANO) POSE + DOFS

grasp_pose = torch.zeros(9)
# grasp_pose[0:3] = torch.tensor([0.1, 0.2, 0.3])
# Identity rotation in 6d rot representation is: (1,0,0,0,1,0)
grasp_pose[3:] = torch.tensor(palm_basis.T.reshape(-1)[:6])
grasp_pose[:3] = torch.tensor(palm_trans)
print("Pose:", grasp_pose)

# grasp_dofs = -1 * torch.tensor(angles) if use_left_hand else torch.tensor(angles)
grasp_dofs = torch.tensor(angles)

print("DOFS:", grasp_dofs)

sample_grasp_q = (
  torch.cat(
    [
      grasp_pose,
      grasp_dofs,
    ]
  )
  .unsqueeze(0)
  .to(device)
  .float()
)

Pose: tensor([ 0.9479,  0.4664,  1.3131, -0.1286, -0.9144,  0.3839, -0.4065, -0.3045,
        -0.8614])
DOFS: tensor([ 0.5058,  0.4457,  0.7103,  0.5521,  0.2008,  0.7043,  0.7848,  0.3649,
        -0.6119,  0.5683,  0.5569,  0.5610, -0.0248,  0.5800,  0.8120,  0.4313,
         1.1895, -0.3507, -0.0896, -0.0561])


In [106]:
grasp_transfer_opt = AdamGraspTransfer(
  source_gripper,
  target_gripper,
  learning_rate=1e-3,
  device=device
)

In [107]:
q_traj, energy, _ = grasp_transfer_opt.run_adam(
  sample_grasp_q.squeeze(0), running_name="test"
)

In [108]:
print(q_traj.shape)
best_q = q_traj[0, -1]
print(best_q.shape)

torch.Size([32, 301, 9])
torch.Size([9])


In [109]:
if best_q.shape[0] != 9 + len(target_model.dynamic_joints):
  # We optimized only for pose, so need to provide dummy joints
  best_q = torch.cat((best_q, target_model.dynamic_joints_q_upper[0]), dim=0)

## Viz Src + Target

In [110]:
print("Plotting TARGET and SOURCE together...")

vis_data = source_model.get_plotly_data(q=sample_grasp_q, color='red')
target_gripper_mesh_data = target_model.get_plotly_data(q=best_q.unsqueeze(0).float().to(device), color='green')
vis_data += target_gripper_mesh_data
fig = go.Figure(data=vis_data)
# fig.show()


Plotting TARGET and SOURCE together...


In [111]:
len(target_gripper_mesh_data)

3

In [112]:
trimesh_list = []

for mesh in target_gripper_mesh_data:
    vertices = np.array([mesh.x, mesh.y, mesh.z]).T
    faces = np.array([mesh.i, mesh.j, mesh.k]).T
    trimesh_list.append(trimesh.Trimesh(vertices=vertices, faces=faces))

combined_mesh = trimesh.util.concatenate(trimesh_list)
combined_mesh.export('mesh.ply')

b'ply\nformat binary_little_endian 1.0\ncomment https://github.com/mikedh/trimesh\nelement vertex 987\nproperty float x\nproperty float y\nproperty float z\nelement face 1962\nproperty list uchar int vertex_indices\nend_header\n\xfe6k?tZ\xe7>8\xb5\x99?\xe9\x95k?\x1d2\xe7>T\xaa\x99?t4k?d\xf2\xde>\\\xb5\x9a?#\x10r?#J\xe4>\x04\x1e\x99?N\rr?{a\xe4>\xa2\x1b\x99?\xd8\xads?V\xa1\xd8>rP\x9a?8Ri?gS\xb5>\x04\x05\xa0?\x17Ri?\x8b\x12\xb5>/\x13\xa0?\x84\xf3h?\xd2\x05\xb6>\x9d\xf1\x9f?\xb4,p?\xdb\t\xb3>\xd9]\xa1?\xed\x01q?Cx\xc8>\xecQ\xac?\xed1j?,\xdf\xb4>\x1b\x03\xa2?\x80\xfbv?!?\xc6>\xddx\xab?\xe9\x80p?\xbd\xf8\xb2>,R\xa1?\xef\x08t?\xb0"\xc6>\r\xf8\x9c?\xb3\xe6r?\x8f!\xbe>\xa2\xce\x9d?l\xc4s?\r\xcb\xc5>\x97\xc3\x9c?\x0f\xcc`?\x99\x05\xcf>\x19\xe4\x9e?V\xf9`?\xe5\\\xd3>\x01\x94\x9e?w\xea`?\xe7\x02\xd3>\rf\x9e?i#a?}\xf4\xd6>u\xe4\x9d?Rza?%?\xca>\x80\xf3\x9e?\xcf\xb6a?~\x9f\xbb>\x80\xa1\xa0?\x02ba?B\xb5\xbb>d\xb9\xa0?v\xf3`?\xbdx\xbc>\xec\x8d\xa2?\x9b\xfe`?k\xc2\xbc>\xfb\xbf\xa2?6\xd1`?\x83\xc6\xbd>\

## Viz Mano + URDF

In [113]:
hand_ply = f"{frame_id}_{int(not use_left_hand)}.ply"
# hand_ply = f"{frame_id}_{1}.ply"

mano_mesh = trimesh.load_mesh(f"../data/{hand_ply}")
print(mano_mesh)

x, y, z = mano_mesh.vertices.T
# i, j, k = mano_mesh.faces.T

vis_data = source_model.get_plotly_data(q=sample_grasp_q, color='red', opacity=0.2)
vis_data += [
        go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(size=2, color='green')
        )
    ]


fig = go.Figure(data=vis_data)
fig.show()
# fig.write_html("gtransfer_test.html")


<trimesh.PointCloud(vertices.shape=(778, 3), name=`000247_0.ply`)>


In [118]:
mano_trans

array([0.84295843, 0.36894829, 1.33611083])

In [122]:
print(data['right'])
print(data['opt_translation'][0])
print(center)


[0 1]
[0.84295843 0.36894829 1.33611083]
[0.73429989 0.46424975 1.33776071]


In [117]:
center

array([0.73429989, 0.46424975, 1.33776071])

In [129]:
verts = np.array(mano_mesh.vertices)
center = np.mean(verts, axis=0)
print(center.shape)

all_verts = verts - mano_trans


cv = verts - center
cv[:, 0] *= -1
# cv[:, 1] *= -1
nv = cv + center
x, y, z = nv.T


all_verts[:, 0] *= -1
all_verts += mano_trans

x, y, z = all_verts.T

# vis_data = []
vis_data = source_model.get_plotly_data(q=sample_grasp_q, color='red', opacity=0.2)
vis_data += [
        go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(size=2, color='blue')
        )
    ]

x,y,z = mano_mesh.vertices.T
vis_data += [
        go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(size=2, color='green')
        )
    ]



fig = go.Figure(data=vis_data)
fig.show()
# fig.write_html("gtransfer_test.html")




(3,)
